In [11]:
import os
import sqlite3
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

# Download Olist dataset via kagglehub
path = kagglehub.dataset_download('olistbr/brazilian-ecommerce')
print(' Dataset downloaded to path:', path)

 Dataset downloaded to path: C:\Users\HP\.cache\kagglehub\datasets\olistbr\brazilian-ecommerce\versions\2


In [12]:
customers = pd.read_csv(os.path.join(path, 'olist_customers_dataset.csv'))
geolocation = pd.read_csv(os.path.join(path, 'olist_geolocation_dataset.csv'))
items = pd.read_csv(os.path.join(path, 'olist_order_items_dataset.csv'))
payments = pd.read_csv(os.path.join(path, 'olist_order_payments_dataset.csv'))
reviews = pd.read_csv(os.path.join(path, 'olist_order_reviews_dataset.csv'))
orders = pd.read_csv(os.path.join(path, 'olist_orders_dataset.csv'))
products = pd.read_csv(os.path.join(path, 'olist_products_dataset.csv'))
sellers = pd.read_csv(os.path.join(path, 'olist_sellers_dataset.csv'))
category_translation = pd.read_csv(
    os.path.join(path, 'product_category_name_translation.csv')
)

print(' All 9 CSV files loaded successfully!')

 All 9 CSV files loaded successfully!


In [15]:
datasets = {
    'customers': customers,
    'orders': orders,
    'items': items,
    'payments': payments,
    'reviews': reviews,
    'products': products,
    'sellers': sellers,
}

for name, df in datasets.items():
  print(
      f'Dataset: {name:<15} | Shape: {df.shape} | Missing Values:'
      f' {df.isnull().sum().sum()}'
  )

print(' Data Quality Checks Completed!')

Dataset: customers       | Shape: (99441, 5) | Missing Values: 0
Dataset: orders          | Shape: (99441, 10) | Missing Values: 7873
Dataset: items           | Shape: (112650, 7) | Missing Values: 0
Dataset: payments        | Shape: (103886, 5) | Missing Values: 0
Dataset: reviews         | Shape: (99224, 7) | Missing Values: 145903
Dataset: products        | Shape: (32951, 10) | Missing Values: 2448
Dataset: sellers         | Shape: (3095, 4) | Missing Values: 0
 Data Quality Checks Completed!


In [18]:
# Convert date columns to datetime
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for col in date_cols:
  if col in orders.columns:
    orders[col] = pd.to_datetime(orders[col])

# Calculate delivery delay in days and flag late status
if (
    "order_delivered_customer_date" in orders.columns
    and "order_estimated_delivery_date" in orders.columns
):
  orders["delivery_delay_days"] = (
      orders["order_delivered_customer_date"]
      - orders["order_estimated_delivery_date"]
  ).dt.total_seconds() / (24 * 3600)
  orders["is_late"] = orders["delivery_delay_days"] > 0

# Safely load category translation and merge
try:
  category_translation = pd.read_csv(
      os.path.join(path, "product_category_name_translation.csv")
  )
  products = products.merge(
      category_translation, on="product_category_name", how="left"
  )
  products["product_category_name_english"] = products[
      "product_category_name_english"
  ].fillna("Unknown")
except Exception as e:
  products["product_category_name_english"] = "Unknown"

# Aggregate reviews and payments per order
review_agg = (
    reviews.groupby("order_id")
    .agg(
        review_score=("review_score", "mean"),
        review_count=("review_id", "count"),
    )
    .reset_index()
)

payment_agg = (
    payments.groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_types=("payment_type", lambda x: ", ".join(x.unique())),
        max_installments=("payment_installments", "max"),
    )
    .reset_index()
)

print(" Data Cleaning & Aggregations Completed Successfully!")

 Data Cleaning & Aggregations Completed Successfully!


In [20]:
master_df = (
    orders.merge(customers, on='customer_id', how='left')
    .merge(payment_agg, on='order_id', how='left')
    .merge(review_agg, on='order_id', how='left')
)

# Assertions for data validation
assert (
    master_df['order_id'].nunique() == len(master_df)
), 'Duplicates found in master table!'
assert master_df['order_id'].notnull().all(), 'Null order IDs in master table!'

print(
    ' Final Order-Level Master Table created successfully! Total Rows:'
    f' {len(master_df)}'
)
display(master_df.head(3))

 Final Order-Level Master Table created successfully! Total Rows: 99441


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_delay_days,is_late,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment_value,payment_types,max_installments,review_score,review_count
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-7.107488,False,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,"credit_card, voucher",1.0,4.0,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-5.355729,False,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,boleto,1.0,4.0,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-17.245498,False,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,credit_card,3.0,5.0,1.0


In [21]:
# Connect to SQLite and write tables
conn = sqlite3.connect('olist_analytics.db')
master_df.to_sql('orders_master', conn, if_exists='replace', index=False)
items.to_sql('order_items', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
sellers.to_sql('sellers', conn, if_exists='replace', index=False)

# Create SQL View
cursor = conn.cursor()
cursor.execute("""
CREATE VIEW IF NOT EXISTS vw_order_summary AS
SELECT 
    m.order_id, m.customer_id, m.customer_unique_id, m.customer_state,
    m.order_status, m.order_purchase_timestamp, m.total_payment_value,
    m.review_score, m.is_late, m.delivery_delay_days
FROM orders_master m;
""")
conn.commit()

# SQL Analysis Q1, Q3, Q4 examples
q1_df = pd.read_sql_query(
    """
    SELECT strftime('%Y-%m', order_purchase_timestamp) AS year_month,
           COUNT(order_id) AS total_orders, SUM(total_payment_value) AS revenue
    FROM vw_order_summary WHERE order_status = 'delivered'
    GROUP BY 1 ORDER BY 1;
""",
    conn,
)

q3_df = pd.read_sql_query(
    """
    SELECT customer_state, COUNT(*) AS total_orders,
           AVG(CASE WHEN is_late = 1 THEN 1.0 ELSE 0.0 END) * 100 AS late_percentage
    FROM vw_order_summary WHERE order_status = 'delivered'
    GROUP BY customer_state ORDER BY late_percentage DESC;
""",
    conn,
)

q4_df = pd.read_sql_query(
    """
    SELECT 
        CASE 
            WHEN delivery_delay_days <= 0 THEN 'On Time / Early'
            WHEN delivery_delay_days BETWEEN 0 AND 3 THEN '1-3 Days Late'
            ELSE '3+ Days Late'
        END AS delay_bucket,
        AVG(review_score) AS avg_review_score, COUNT(*) AS total_orders
    FROM vw_order_summary WHERE review_score IS NOT NULL AND order_status = 'delivered'
    GROUP BY 1;
""",
    conn,
)

print(' SQLite Database, Views & SQL Analysis Queries Executed Successfully!')
display(q3_df.head(3))

 SQLite Database, Views & SQL Analysis Queries Executed Successfully!


,customer_state,total_orders,late_percentage
0,AL,397,23.929471
1,MA,717,19.665272
2,PI,476,15.966387


In [24]:
# Correlation Matrix
corr_matrix = master_df[
    ['delivery_delay_days', 'review_score', 'total_payment_value']
].corr()
print('Correlation Matrix:\n', corr_matrix)

# Findings Summary
total_rev = master_df['total_payment_value'].sum()
late_rate = master_df['is_late'].mean() * 100
print('\n=== OLIST E-COMMERCE ANALYSIS FINDINGS SUMMARY ===')
print(f'1. Total Revenue Processed: ${total_rev:,.2f}')
print(f'2. Overall Late Delivery Rate: {late_rate:.2f}%')
print(
    '3. Key Insight: Shipping delays heavily degrade customer review scores.'
)

# Export for Power BI
master_df.to_csv('olist_powerbi_master.csv', index=False)
q3_df.to_csv('state_performance_powerbi.csv', index=False)
print(
    '\n All clean CSV files exported successfully for Power BI dashboard'
    ' integration!'
)

Correlation Matrix:
                      delivery_delay_days  review_score  total_payment_value
delivery_delay_days             1.000000     -0.266643            -0.017969
review_score                   -0.266643      1.000000            -0.049137
total_payment_value            -0.017969     -0.049137             1.000000

=== OLIST E-COMMERCE ANALYSIS FINDINGS SUMMARY ===
1. Total Revenue Processed: $16,008,872.12
2. Overall Late Delivery Rate: 7.87%
3. Key Insight: Shipping delays heavily degrade customer review scores.

 All clean CSV files exported successfully for Power BI dashboard integration!


In [25]:
import os

# Yeh current folder ka path print karega aur folder window khol dega
print("Folder Path:", os.getcwd())
os.startfile(os.getcwd())

Folder Path: C:\Users\HP
